# Multi-Agent Systems

The longer and larger tasks we give our agent, the more its performance may start to suffer. For example, let's say you want an agent to produce a high-quality market research report. This task is long and requires several _specialized_ agents at different times, like proposing an outline, researching on the web, validating the resources, writing the report, and finally editing the final report.

If you try to hand this all off to a single agent, then it would have too many tools to choose from, their context window would keep overflowing with information, and the quality of the output would suffer as a result.

Conversely, a _multi-agent system_ would break this complex application down into several _specialized_ agents that work together to solve the problem, rather than relying on a single agent to handle every step. 

When developers are looking for multi-agent systems, they are often wanting to address the following issues:
* **Context management:** Provide specialized knowledge without overwhelming the model’s context window. If context were infinite and latency zero, you could dump all knowledge into a single prompt - but since it’s not, you need patterns to selectively surface relevant information.
* **Distributed development:** Allow different teams to develop and maintain capabilities independently, composing them into a larger system with clear boundaries. 
* **Parallelization:** Spawn specialized workers for subtasks and execute them concurrently for faster results.

When _tasks require specialized knowledge with extensive context_ (long prompts and domain-specific tools), or when you _need to enforce sequential constraints_ that unlock capabilities only after certain conditions are met.

## Patterns for building Multi-Agent Systems
There are several _patterns_ for building multi-agent systems, each suited to different use cases:

| Pattern | How it Works| Workflow |
| :-- | :-- | :--: |
| Sub-Agents | A main agent coordinates subagents as tools. All routing passes through the main agent, which decides when and how to invoke each subagent. | <div align="left"> <img src="images/pattern-subagents.avif" width="250" heigh="250" alt="Sub Agents"/> </div> |
| Handoffs | Behavior changes dynamically based on state. Tool calls update a state variable that triggers routing or configuration changes, switching agents or adjusting the current agent’s tools and prompt. | <div align="left"> <img src="images/pattern-handoffs.avif" width="250" heigh="200" alt="Handoffs"/> </div> |
| Handoffs | Behavior changes dynamically based on state. Tool calls update a state variable that triggers routing or configuration changes, switching agents or adjusting the current agent’s tools and prompt. | <div align="left"> <img src="images/pattern-handoffs.avif" width="250" heigh="200" alt="Handoffs"/> </div> |
| Skills | Specialized prompts and knowledge loaded on-demand. A single agent stays in control while loading context from skills as needed. | <div align="left"><img src="images/pattern-skills.avif" width="200" heigh="200" alt="Skills"/> </div> |
| Router | A routing step classifies input and directs it to one or more specialized agents. Results are synthesized into a combined response. | <div align="left"> <img src="images/pattern-router.avif" width="250" heigh="250" alt="Router"/></div> |
| Custom Workflow | Build bespoke execution flows with LangGraph, mixing deterministic logic and agentic behavior. Embed other patterns as nodes in your workflow. | &nbsp; |

## Simple example of a sub-agents application
Here is a very simple example of a sub-agents application - we will create a more practical example later in this notebook.

In [1]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

Now let's create 2 simple agents, each with 1 tool. Notice that each agent is a specialized agent (i.e. designed to do a specialized, though for this example, a very simple task.

In [2]:
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
# a simple square-root agent, with attached tool
@tool
def square_root(num: float) -> float:
    """returns the square root of a number"""
    print(f" ----- calling square_root({num:.3f}) tool -----")
    return num**0.5


square_root_agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=(
        """You are a math-wiz that can calculate square-roots of numbers using the 
        square_root tool provided to you. Use only this tool to calculate square-roots
        and return just the result and no extra text"""
    ),
    tools=[square_root],
)

# test this agent
response = square_root_agent.invoke(
    {"messages": {"role": "user", "content": "What is the square root of 23467?"}}
)
print(response["messages"][-1].content)

 ----- calling square_root(23467.000) tool -----
153.18942522250026


In [6]:
# a simple squares agent, with attached tool
@tool
def square(num: float) -> float:
    """returns the square of a number"""
    print(f" ----- calling square({num:.3f}) tool -----")
    return num**2


square_agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=(
        """You are a math-wiz that can calculate squares of numbers using the 
        square tool provided to you. Use only this tool to calculate squares
        and return just the result and no extra text"""
    ),
    tools=[square],
)

# test this agent
response = square_agent.invoke(
    {"messages": {"role": "user", "content": "What is the square of 153.19?"}}
)
print(response["messages"][-1].content)

 ----- calling square(153.190) tool -----
23467.1761


In the sub-agents pattern, an agent acts as a tool of the "super" agent. We wrap the `agent.invoke()` call in a tool function, which effectively makes an agent a tool. Let's see how that works.

In [9]:
@tool
def call_subagent_square_root(num: float) -> float:
    """calls square_root agent as a tool to calculate square roots of numbers"""
    print(f" ----- calling call_subagent_square_root({num:.3f}) tool -----")
    response = square_root_agent.invoke(
        {"messages": {"role": "user", "content": f"What is the square root of {num}"}}
    )
    return response["messages"][-1].content


@tool
def call_subagent_square(num: float) -> float:
    """calls square agent as a tool to calculate squares of numbers"""
    print(f" ----- calling call_subagent_square({num:.3f}) tool -----")
    response = square_agent.invoke(
        {"messages": {"role": "user", "content": f"What is the square of {num}"}}
    )
    return response["messages"][-1].content

Now we create our main agent, which with the above tool calls as tools, which effectively makes this the **sub-agent pattern** for calling agents.

In [10]:
main_agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=(
        """You are a helpful agent that can calculate squares an square-roots of numbers
        by calling sub-agents assigned to you. You have access to the following sub-agents
        as tools:
        - call_subagent_square_root: to calculate square roots
        - call_subagent_square: to calculate squares 
        Use only these tools for the square-root and squares calculations respectively.
        Return just the calculated number and no extra text"""
    ),
    tools=[call_subagent_square_root, call_subagent_square],
)

In [11]:
response = main_agent.invoke(
    {"messages": {"role": "user", "content": "What is the square root of 23467?"}}
)
print(response["messages"][-1].content)

 ----- calling call_subagent_square_root(23467.000) tool -----
 ----- calling square_root(23467.000) tool -----
153.18942522250026


In [12]:
response = main_agent.invoke(
    {"messages": {"role": "user", "content": "What is the square of 153.19?"}}
)
print(response["messages"][-1].content)

 ----- calling call_subagent_square(153.190) tool -----
 ----- calling square(153.190) tool -----
23467.1761


Perfect! Now let's move onto a more practical example of a wedding planner.

### Summary 
In the **subagents** architecture, a central main agent (often referred to as a supervisor) coordinates subagents by _calling them as tools_. The main agent decides which subagent to invoke, what input to provide, and how to combine results. _Subagents are stateless_ — they don’t remember past interactions, with all conversation memory maintained by the main agent. This provides context isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.

#### Key characteristics
* **Centralized control:** All routing passes through the main agent
* **No direct user interaction:** Subagents return results to the main agent, not the user (though you can use interrupts within a subagent to allow user interaction)
* **Subagents via tools:** Subagents are invoked via tools
* **Parallel execution:** The main agent can invoke multiple subagents in a single turn

#### When to use
Use the subagents pattern when you have multiple distinct domains (e.g., calendar, email, CRM, database), subagents don’t need to converse directly with users, or you want centralized workflow control. For simpler cases with just a few tools, use a single agent.

---

## Wedding Planner - sub-agents pattern example
Planning a wedding is not easy - you have to plan for venues (destinations), transportation, catering, music (and hope the bride & groom actually show up 😄!). Jokes apart, this is a perfect use-case for the sub-agents pattern we just discussed above.

Let's build a **Destination Wedding Planner Dream Team**, which consists of at least:
- A **flights agent** that can find flights to and from your destination
- A **venue agent** that searches the web for a wedding venue at your destination
- A **DJ agent** that scours the web for music  matching a certain genre
- And the main **co-ordinator agent** that co-ordinates all these sub-agents

Here is what our agents will look like:

<div align="left"> <img src="images/wedding_planner_agents.png" width="350" heigh="350" alt="Wedding Planner Sub Agents"/> </div>

**High Level Design of sub-agents**
- The **flights agent** will be connected to the Kiwi MCP server to find flights
- The **venue agent** will be connected to the web-search tool 
- The **DJ agent** will be paired with a SQL query tool, which queries our local database for music
- The **co-ordinator agent** will have a few jobs: get a few key wedding details `date of wedding`, `origin`, `destination`, `guest count`, and preferred music `genre` (for simplicity let's assume all guests invited for the wedding are from the same `origin`).

As a first step, let's setup the tools for our **Wedding Planning Dream Team**

(**NOTE:** this section of the notebook is designed as an _independent_ section, so some imports repeat!)

In [1]:
import sys
import asyncio
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In [2]:
# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

#### Step1: Create tools for our agents

In [3]:
# tools for the Flights agent

from langchain_mcp_adapters.client import MultiServerMCPClient

# an yet another free flights MCP server
kiwi_client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

kiwi_tools = await kiwi_client.get_tools()
print("-------- kiwi_ckient -> list of tools available -------- ")
console.print(kiwi_tools)

-------- kiwi_ckient -> list of tools available -------- 


[
    StructuredTool(
        name='search-flight',
        description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights
between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to
resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this 
tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just 
city names.\n\nYou should display the returned results in a markdown table format: Group the results by price 
(those who are the cheapest), duration (those who are the shortest, i.e. have the smallest 
\'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight
in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → 
Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates & times in the local 
timezones, and duration of the flight (e.g. "03/08 06:05 → 09:30 (3h 25m)", use \'durationInSeconds\' to display 
the duration and not \'totalDurationInSeconds\')\n  - In the 3rd column: The cabin class (e.g. "Economy")\n  - (In 
case of return flight only) In the 4th column: The return flight departure and arrival airports, including layovers
(e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - (In case of return flight only) In the 5th column: The return 
flight departure and arrival dates & times in the local timezones, and duration of the flight (e.g. "03/08 06:05 → 
09:30 (3h 25m)", use \'return.durationInSeconds\' to display the duration)\n  - (In case of return flight only) In 
the 6th column: The return flight cabin class (e.g. "Economy")\n  - In the previous-to-last column: The total price
of the flight\n  - In the last column: The deep link to book the flight\n\nFinally, provide a summary highlighting 
the best prices, the shortest flights and a recommendation. End wishing a nice trip to the user with a short fun 
fact about the destination!\n',
        args_schema={
            'type': 'object',
            'properties': {
                'flyFrom': {
                    'type': 'string',
                    'minLength': 1,
                    'description': 'Location to fly from: It could be a city or an airport name or code'
                },
                'flyTo': {
                    'type': 'string',
                    'description': 'Location to fly to: It could be a city or an airport name or code'
                },
                'departureDate': {
                    'type': 'string',
                    'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$',
                    'description': 'Departure date in dd/mm/yyyy format'
                },
                'departureDateFlexRange': {
                    'type': 'integer',
                    'minimum': 0,
                    'maximum': 3,
                    'default': 0,
                    'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected
departure date)'
                },
                'returnDate': {
                    'type': 'string',
                    'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$',
                    'description': 'Return date in dd/mm/yyyy format'
                },
                'returnDateFlexRange': {
                    'type': 'integer',
                    'minimum': 0,
                    'maximum': 3,
                    'default': 0,
                    'description': 'Return date flexibility range in days (0 to 3 days before/after the selected 
return date)'
                },
                'passengers': {
                    'type': 'object',
                    'properties': {
                        'adults': {
                            'type': 'integer',
                            'minimum': 0,
                            'maximum': 9,


In [5]:
# tools for the venue agent
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information
    Args:
        query (str): the search query
    Returns:
        Dict[str, Any]: the search results
    """
    print(f"------ Calling web_search({query}) tool ------")
    return tavily_client.search(query)

In [6]:
# music genre search - we'll search a local SQLite database, specifically the Chinook db
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///db/chinook.db")


@tool
def query_playlist_db(query: str) -> str:
    """query database for playlist information"""
    print(f"------ Calling query_playlist_db({query}) tool ------")
    try:
        return db.run(query)
    except Exception as ex:
        return f"Error querying database {ex}"

#### Step2: Create Agent state
Next, let's create an `AgentState` that is shared amongst our agents. Sub-agents can read this and the co-ordinator agent can update this state. Thus we can share data between agents.

In [7]:
from langchain.agents import AgentState


class WeddingState(AgentState):
    origin: str
    destination: str
    # NOTE: pls use dd/mm/yyyy format only for dates!
    wedding_date: str
    guest_count: str
    genre: str

#### Step3: Create our sub-agents and co-ordinator agent

In [8]:
from langchain.agents import create_agent

In [9]:
# travel agent
travel_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=kiwi_tools,
    system_prompt=(
        """You are a travel agent. Search for flights from origin to the desired destination wedding location.
        You are not allowed to ask any follow-up questions, you must find the best flight options based on the following:
        - Price (lowest, economy class ONLY)
        - Duration (shortest)
        - Date (time of year when you believe is the best for a wedding at this location)
        To make things easy, look for one ticket, one-way.
        You may need to make multiple searches to iteratively find the best flight options.
        You will be given no extra information, only the origin and destination. It is your job to think
        critically about the flight options.
        Once you have found the best options, let the user know your shortlist of options."""
    ),
)

In [10]:
venue_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[web_search],
    system_prompt=(
        """You are a venue specialist. Search for venues at the desired location and with desired capacity.
        You are not allowed to ask any follow-up questions. You must find the best options based on the following:
        - Price (lowest)
        - Capacity (exact match)
        - Reviews (highest)
        You may need to make multiple searches to iteratively find the best options.
        """
    ),
)

In [11]:
playlist_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt=(
        """You are a playlist specialist. Query the SQL database and curate the best playlist for a wedding given a genre. Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price. 
        If you run into errors when querying the database, try to fix them by making changes to the query.
        Do not come back empty-handed, keep trying to query the database until you find a list of songs.
        You may need to make multiple queries to iteratively find the best options.
        """
    ),
)

#### Step4: create the co-ordinator (boss agent!)

In [12]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    print(
        f"------ Calling search_flights(origin={origin}, destination={destination}) tool ------"
    )
    response = await travel_agent.ainvoke(
        {
            "messages": [
                HumanMessage(content=f"Find flights from {origin} to {destination}")
            ]
        }
    )
    return response["messages"][-1].content


@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    print(
        f"------ Calling search_venues(destination={destination}, capacity={capacity}) tool ------"
    )
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content


@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    print(f"------ Calling suggest_playlist(genre={genre}) tool ------")
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content


@tool
def update_state(
    origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime
) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    print(
        f"------ Calling update_state(origin={origin},destination={destination},guest_count={guest_count},genrs={genre}) tool ------"
    )
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "genre": genre,
            "messages": [
                ToolMessage(
                    "Successfully updated state", tool_call_id=runtime.tool_call_id
                )
            ],
        }
    )

In [13]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """,
)

#### Step5: Ready? Let's go!!

> **Warning!!**
>
> This query will run for a long time - be patient!<br/><br/>

In [14]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="I'm from Mumbai and I'd like a wedding in Lake Como, Italy for 200 guests, jazz-genre"
            )
        ],
    },
    config={
        "tags": ["WP"],
        "recursion_limit": 40,
    },
    # tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

------ Calling update_state(origin=Mumbai,destination=Lake Como, Italy,guest_count=200,genrs=jazz) tool ------
------ Calling search_flights(origin=Mumbai, destination=Lake Como, Italy) tool ------
------ Calling search_venues(destination=Lake Como, Italy, capacity=200) tool ------
------ Calling suggest_playlist(genre=jazz) tool ------
------ Calling query_playlist_db(SELECT track_id, title, artist, duration_sec, price FROM tracks WHERE genre = 'Jazz' AND wedding_friendly = TRUE ORDER BY popularity DESC LIMIT 30;) tool ------
------ Calling web_search(Lake Como wedding venue 200 guests capacity) tool ------
------ Calling query_playlist_db(SELECT id, title, artist, duration_sec, price FROM tracks WHERE genre = 'Jazz' AND wedding_friendly = TRUE ORDER BY popularity DESC LIMIT 30;) tool ------
------ Calling query_playlist_db(SELECT track_id AS track_id, title, artist, duration_sec, price FROM tracks WHERE genre = 'Jazz' AND wedding_friendly = TRUE ORDER BY popularity DESC LIMIT 30;) to

In [33]:
from pprint import pprint

# show me all the messages exchanged!
pprint(response)

{'destination': 'Lake Como, Italy',
 'genre': 'jazz',
 'guest_count': '200',
 'messages': [HumanMessage(content="I'm from Mumbai and I'd like a wedding in Lake Como, Italy for 200 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='628262a7-aab6-42d7-a316-846742184b17'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 490, 'prompt_tokens': 339, 'total_tokens': 829, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWyQNyhGjjMk7HLXKLgDEgsEQJpuo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dae9e-d52d-7631-a177-1fa08750ef69-0', tool_calls=[{'name': 'update_state', 'args': {'origin

In [36]:
# let's inspect teh final response
print(response["messages"][-1].content)

Wonderful — we’ve got a solid starting plan for a 200-guest jazz-infused Lake Como wedding. Here’s a concise recap of what I’ve gathered and the next steps I can take for you.

What I’ve locked in so far
- Origin: Mumbai, India
- Destination: Lake Como, Italy (best access via Milan MXP, LIN, or BGY)
- Guest count: 200
- Vibe/genre: Jazz (romantic, refined lounge/jazz standards with a touch of bossa nova)

Flight options (for guests traveling to Milan area)
- Cheapest overall: BOM → MXP via AUH
  - Price: ~€220
  - Duration: ~23h15m
  - Pros: Lowest cost
  - Cons: Very long travel day with a long layover
  - Link: booking option available (example: Kiwi link)
- Good value, mid-range: BOM → MXP via DOH or SHJ, ~€298–€325
  - Duration: 12–15+ hours depending on routing
  - Pros: More balanced travel time and price
- Fastest option (for guests who want minimal travel time): BOM → LIN via MUC
  - Price: ~€876
  - Duration: ~11 hours total
  - Pros: Quickest route to the Lombardy region; eas